# Browse Processed NetCDF Outputs

Select a folder of processed `.nc` files, inspect the detected schema, and plot individual measurements.

Supported FCS outputs:

- standard grid: `time x cutoff x deadband`
- full MCMC grid: `time x cutoff x deadband x MC`
- best Pareto MCMC: `time x MC`, optionally with `best_deadband` and `best_cutoff`


## Setup


In [ ]:
import pathlib
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display
except ImportError:
    widgets = None
    clear_output = None
    display = print

warnings.filterwarnings("ignore")

NOTEBOOK_DIR = pathlib.Path.cwd()
for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break
else:
    PROJECT_ROOT = pathlib.Path.cwd()

POST_PROCESSING_DIR = PROJECT_ROOT / "notebooks" / "post_processing"
DEFAULT_DATA_DIR = POST_PROCESSING_DIR / "data"
DEFAULT_SMOKE_DIR = PROJECT_ROOT / "notebooks" / "processing" / "output" / "smoke_test"


## Helper Functions


In [ ]:
def find_netcdf_files(folder, pattern="*.nc"):
    folder = pathlib.Path(folder).expanduser()
    if not folder.exists():
        raise FileNotFoundError(f"Folder does not exist: {folder}")
    files = sorted(folder.glob(pattern))
    return [path for path in files if path.is_file()]


def open_processed_dataset(files):
    files = [pathlib.Path(path) for path in files]
    if not files:
        raise ValueError("Select at least one NetCDF file.")
    if len(files) == 1:
        return xr.open_dataset(files[0])
    try:
        return xr.open_mfdataset(files, combine="by_coords").sortby("time")
    except Exception as exc:
        schemas = [inspect_netcdf_file(path) for path in files]
        raise ValueError(
            "Selected files could not be combined by coordinates. "
            "Choose files with the same schema and compatible coordinates."
        ) from exc


def classify_dataset(ds):
    dims = set(ds.sizes)
    if {"time", "cutoff", "deadband", "MC"}.issubset(dims):
        return "full_mcmc_grid"
    if {"time", "cutoff", "deadband"}.issubset(dims):
        return "standard_grid"
    if {"time", "MC"}.issubset(dims):
        return "best_pareto_mcmc"
    if "time" in dims:
        return "time_series"
    return "unknown"


def inspect_netcdf_file(path):
    path = pathlib.Path(path)
    with xr.open_dataset(path) as ds:
        return {
            "file": path.name,
            "schema": classify_dataset(ds),
            "dims": ", ".join(f"{name}={size}" for name, size in ds.sizes.items()),
            "variables": ", ".join(ds.data_vars),
        }


def schema_table(files):
    return pd.DataFrame([inspect_netcdf_file(path) for path in files])


def variable_options(ds):
    preferred = ["dcdt(HM)", "dcdt(linear)", "AIC(HM)", "RMSE(HM)", "R2(HM)", "nRMSE(HM)"]
    available = list(ds.data_vars)
    ordered = [name for name in preferred if name in available]
    ordered.extend([name for name in available if name not in ordered])
    return ordered


def time_options(ds):
    if "time" not in ds.coords:
        return []
    values = pd.to_datetime(ds["time"].values)
    return [(str(value), value.to_datetime64()) for value in values]


def coordinate_options(ds, coord):
    if coord not in ds.coords:
        return []
    return [(str(value.item() if hasattr(value, "item") else value), value.item() if hasattr(value, "item") else value)
            for value in ds[coord].values]


def _select_time(ds, time_value):
    if "time" not in ds.coords or time_value is None:
        return ds
    return ds.sel(time=time_value)


def _finite_values(values):
    arr = np.asarray(values, dtype=float).ravel()
    return arr[np.isfinite(arr)]


def plot_processed_selection(ds, variable="dcdt(HM)", time_value=None, cutoff=None, deadband=None):
    if variable not in ds:
        raise ValueError(f"Variable {variable!r} is not present in the dataset.")

    schema = classify_dataset(ds)
    selected = _select_time(ds, time_value)
    data = selected[variable]

    if schema == "standard_grid":
        fig, ax = plt.subplots(figsize=(7, 4.5), dpi=120)
        grid = data.transpose("cutoff", "deadband")
        im = ax.imshow(
            grid.values,
            origin="lower",
            aspect="auto",
            extent=[
                float(ds["deadband"].min()),
                float(ds["deadband"].max()),
                float(ds["cutoff"].min()),
                float(ds["cutoff"].max()),
            ],
        )
        ax.set_xlabel("Deadband [s]")
        ax.set_ylabel("Cutoff [s]")
        ax.set_title(f"{variable} at {pd.to_datetime(selected.time.values)}")
        fig.colorbar(im, ax=ax, label=variable)
        fig.tight_layout()
        return fig

    if schema == "full_mcmc_grid":
        median_grid = data.median(dim="MC").transpose("cutoff", "deadband")
        fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=120)
        im = axes[0].imshow(
            median_grid.values,
            origin="lower",
            aspect="auto",
            extent=[
                float(ds["deadband"].min()),
                float(ds["deadband"].max()),
                float(ds["cutoff"].min()),
                float(ds["cutoff"].max()),
            ],
        )
        axes[0].set_xlabel("Deadband [s]")
        axes[0].set_ylabel("Cutoff [s]")
        axes[0].set_title(f"Median {variable}")
        fig.colorbar(im, ax=axes[0], label=variable)

        cutoff = cutoff if cutoff is not None else ds["cutoff"].values[0]
        deadband = deadband if deadband is not None else ds["deadband"].values[0]
        posterior = _finite_values(data.sel(cutoff=cutoff, deadband=deadband).values)
        axes[1].hist(posterior, bins=30, color="steelblue", alpha=0.8)
        axes[1].set_title(f"Posterior cutoff={cutoff}, deadband={deadband}")
        axes[1].set_xlabel(variable)
        axes[1].set_ylabel("Count")
        fig.tight_layout()
        return fig

    if schema == "best_pareto_mcmc":
        dcdt = ds[variable]
        median = dcdt.median(dim="MC")
        q16 = dcdt.quantile(0.16, dim="MC")
        q84 = dcdt.quantile(0.84, dim="MC")
        fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=120)
        times = pd.to_datetime(ds["time"].values)
        axes[0].fill_between(times, q16.values, q84.values, color="steelblue", alpha=0.25, label="16-84%")
        axes[0].plot(times, median.values, color="steelblue", marker="o", ms=3, lw=1.0, label="median")
        axes[0].set_xlabel("Time")
        axes[0].set_ylabel(variable)
        axes[0].legend()

        selected_posterior = _finite_values(_select_time(ds, time_value)[variable].values)
        axes[1].hist(selected_posterior, bins=30, color="steelblue", alpha=0.8)
        axes[1].set_title(f"Selected measurement posterior")
        axes[1].set_xlabel(variable)
        axes[1].set_ylabel("Count")
        fig.autofmt_xdate()
        fig.tight_layout()
        return fig

    if "time" in ds[variable].dims:
        fig, ax = plt.subplots(figsize=(8, 4), dpi=120)
        ax.plot(pd.to_datetime(ds["time"].values), ds[variable].values, marker="o", lw=1)
        ax.set_xlabel("Time")
        ax.set_ylabel(variable)
        ax.set_title(variable)
        fig.autofmt_xdate()
        fig.tight_layout()
        return fig

    raise ValueError(f"Unsupported dataset schema: {schema}")


## Plain Python Example


In [ ]:
example_folder = DEFAULT_SMOKE_DIR if DEFAULT_SMOKE_DIR.exists() else DEFAULT_DATA_DIR
example_files = find_netcdf_files(example_folder, "*.nc")

if example_files:
    example_ds = open_processed_dataset(example_files[:1])
    display(schema_table(example_files[:1]))
    display(plot_processed_selection(example_ds, variable_options(example_ds)[0], time_options(example_ds)[0][1]))
else:
    print(f"No .nc files found in {example_folder}. Add files or use the widget browser below.")


## Interactive Browser


In [ ]:
if widgets is None:
    raise ImportError("ipywidgets is required for the interactive browser.")

folder_widget = widgets.Text(
    value=str(DEFAULT_SMOKE_DIR if DEFAULT_SMOKE_DIR.exists() else DEFAULT_DATA_DIR),
    description="Folder",
    layout=widgets.Layout(width="85%"),
)
pattern_widget = widgets.Text(value="*.nc", description="Pattern", layout=widgets.Layout(width="35%"))
refresh_button = widgets.Button(description="Refresh", button_style="primary")
select_all_widget = widgets.Checkbox(value=True, description="Select all")
files_widget = widgets.SelectMultiple(options=[], description="Files", rows=8, layout=widgets.Layout(width="85%"))
variable_widget = widgets.Dropdown(options=[], description="Variable")
time_widget = widgets.Dropdown(options=[], description="Time")
cutoff_widget = widgets.Dropdown(options=[], description="Cutoff")
deadband_widget = widgets.Dropdown(options=[], description="Deadband")
plot_button = widgets.Button(description="Plot", button_style="success")
output = widgets.Output()


def _refresh_files(_=None):
    with output:
        clear_output()
        try:
            files = find_netcdf_files(folder_widget.value, pattern_widget.value)
            options = [(path.name, str(path)) for path in files]
            files_widget.options = options
            if select_all_widget.value:
                files_widget.value = tuple(value for _, value in options)
            display(schema_table(files) if files else pd.DataFrame(columns=["file", "schema", "dims", "variables"]))
        except Exception as exc:
            print(exc)


def _load_selected(_=None):
    with output:
        clear_output()
        try:
            selected_files = [pathlib.Path(value) for value in files_widget.value]
            ds = open_processed_dataset(selected_files)
            print(f"Detected schema: {classify_dataset(ds)}")
            display(schema_table(selected_files))
            variable_widget.options = variable_options(ds)
            if variable_widget.options:
                variable_widget.value = "dcdt(HM)" if "dcdt(HM)" in variable_widget.options else variable_widget.options[0]
            time_widget.options = time_options(ds)
            if time_widget.options:
                time_widget.value = time_widget.options[0][1]
            cutoff_widget.options = coordinate_options(ds, "cutoff")
            deadband_widget.options = coordinate_options(ds, "deadband")
            if cutoff_widget.options:
                cutoff_widget.value = cutoff_widget.options[0][1]
            if deadband_widget.options:
                deadband_widget.value = deadband_widget.options[0][1]
            display(plot_processed_selection(
                ds,
                variable=variable_widget.value,
                time_value=time_widget.value if time_widget.options else None,
                cutoff=cutoff_widget.value if cutoff_widget.options else None,
                deadband=deadband_widget.value if deadband_widget.options else None,
            ))
        except Exception as exc:
            print(exc)


refresh_button.on_click(_refresh_files)
plot_button.on_click(_load_selected)

controls = widgets.VBox([
    widgets.HBox([folder_widget]),
    widgets.HBox([pattern_widget, refresh_button, select_all_widget]),
    files_widget,
    widgets.HBox([variable_widget, time_widget, cutoff_widget, deadband_widget, plot_button]),
    output,
])
display(controls)
_refresh_files()
